# ICU-SEWS · 최종 test 환자 단위 bootstrap 95% 신뢰구간
**CPU 런타임으로 충분합니다. GPU·재학습·원본 데이터 준비는 필요 없습니다.**
최종 학습을 완료한 Drive의 예측 파일을 읽어 위에서부터 실행하세요.

- test 환자 N명 중 N명을 복원추출하며, 선택된 환자의 전체 시간 기록을 함께 반영합니다.
- 같은 환자가 두 번 뽑히면 환자별 경보·Utility·시간별 예측도 두 번 반영합니다.
- 모델·Platt·임계값은 고정합니다. 신뢰구간을 보고 test에 맞춰 조정하지 않습니다.
- 기본 **2,000회, seed 42, percentile 2.5%~97.5%**. 병원/양성 비율을 고정하지 않는 일반 환자 bootstrap입니다.
- 원래 test 점수를 다시 계산해 저장된 점수와 일치하는지 검사합니다.
- 원본 최종 결과를 수정하지 않고 별도 폴더에 저장합니다.
- 현재 고정 모델에 대한 test 환자 표본의 불확실성입니다. 재학습·모델 선택 변동이나 새로운 병원으로의 일반화 불확실성은 포함하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK_ROOT = Path('/content/drive/MyDrive/SEWS_GRU')
RUN_NAME = 'gru_v2_platt_final'
FINAL_RUN = WORK_ROOT/'final'/RUN_NAME
N_BOOTSTRAP = 2000
SEED = 42
files = ['test_predictions.parquet','test_summary.json','frozen.json','platt.json','threshold.json']
for name in files:
    assert (FINAL_RUN/name).is_file(), f'파일이 없습니다: {FINAL_RUN/name}'
print('읽을 최종 결과:', FINAL_RUN)

## 1. 독립적인 Python 3.10 평가 환경
최종 학습용 저장소·코드 버전 기록·모델 파일은 변경하지 않습니다.

In [ ]:
import subprocess
import sys
import shutil
REPO_URL = 'https://github.com/sogang-icu-sews/ICU-SEWS.git'
BRANCH = 'codex/gru-colab'
REPO = Path('/content/ICU-SEWS-GRU-bootstrap')
def run(command, log_path=None):
    from collections import deque
    from contextlib import nullcontext
    tail = deque(maxlen=80)
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
    log_context = log_path.open("w", encoding="utf-8") if log_path is not None else nullcontext(None)
    with log_context as log:
        with subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace") as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                tail.append(line)
                if log is not None:
                    log.write(line)
                    log.flush()
            returncode = process.wait()
    if returncode:
        details = "".join(tail)
        raise RuntimeError(f"실행 실패 (종료 코드 {returncode}). 로그: {log_path}\n마지막 출력:\n{details}")
    return subprocess.CompletedProcess([str(x) for x in command], returncode)

if not REPO.exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,REPO])
else:
    origin = subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()
    assert origin == REPO_URL
    run(['git','-C',REPO,'pull','--ff-only'])
run([sys.executable,'-m','pip','install','-q','uv'])
UV = [sys.executable,'-m','uv']
PY = Path('/content/sews-bootstrap-py310/bin/python')
if not PY.exists():
    run(UV+['venv','--python','3.10',str(PY.parent.parent)])
run(UV+['pip','install','--python',str(PY),'-r',str(REPO/'requirements-gru-eval.txt')])
run([PY,'-c',"import sys; assert sys.version_info[:2]==(3,10); print(sys.version)"])

## 2. 환자 단위 bootstrap 실행
100회마다 진행 상황을 표시합니다. 중단되면 이 셀을 다시 실행해 새 폴더에서 처음부터 계산합니다.
최종 test 추론은 다시 실행하지 않으며 저장된 예측만 사용합니다.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import json

CACHE = Path('/content/sews-bootstrap-inputs')
CACHE.mkdir(parents=True,exist_ok=True)
for name in files:
    shutil.copyfile(FINAL_RUN/name,CACHE/name)
stamp = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
OUT = WORK_ROOT/'confidence_intervals'/f'{RUN_NAME}_{stamp}'
LOG = OUT.parent/'logs'/f'{OUT.name}.log'
run([PY,'-u',REPO/'scripts/bootstrap_gru.py','--run',CACHE,'--out',OUT,
     '--repetitions',str(N_BOOTSTRAP),'--seed',str(SEED)],log_path=LOG)
print('신뢰구간 저장:',OUT)

## 3. 결과 확인
`estimate`는 원래 test 점수, `ci_lower`~`ci_upper`는 95% 신뢰구간입니다.
비율 지표는 0~1 단위이며 0.10은 10%입니다. Utility는 정확도가 아닙니다.
`valid_replicates`가 요청 횟수보다 적으면 해당 지표를 정의할 수 없는 재추출 표본이 있었음을 뜻합니다.

병원별 결과와 60시간 이내 확률 지표에도 같은 전체 환자 추출 횟수를 적용합니다.
60시간 이내 경보 지표는 환자의 전체 기록이 필요하므로 계산하지 않습니다.
표의 각 구간은 개별 95% 구간이며 모든 지표를 동시에 포함하는 구간은 아닙니다.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

if 'OUT' not in globals():
    # 결과만 볼 때는 첫 Drive 연결 셀과 이 셀을 실행하세요.
    completed = sorted((WORK_ROOT/'confidence_intervals').glob(f'{RUN_NAME}_*/metadata.json'))
    assert completed, '완료된 신뢰구간 결과가 없습니다.'
    OUT = completed[-1].parent
metadata = json.loads((OUT/'metadata.json').read_text())
assert metadata['complete']
ci = pd.read_csv(OUT/'bootstrap_95ci.csv')
print('읽는 결과:',OUT)
main = ['AUROC','AUPRC','Brier','Brier_skill','row_sensitivity','row_PPV',
        'nonsepsis_patient_false_alarm_rate','early_recall_6to12h',
        'pre_onset_recall_0to12h','median_lead_hours_detected','Utility']
print('① 전체 test의 핵심 지표와 95% 신뢰구간')
display(ci[(ci.scope=='all') & ci.metric.isin(main)].round(5))
print('② 병원별·60시간 이내')
display(ci[(ci.scope!='all') & ci.metric.isin(main)].round(5))
print('공유할 파일:',OUT/'bootstrap_95ci.csv')